# 주제 ③ 프레스 유압펌프 진동·전류 시계열 — 추가 진단 (02_diagnosis_deep)

- 목적: `01_data_quality.ipynb`의 주장 중 근거가 약했던 항목을 검증하고, 모델 단계에서 바로 쓸 세그먼트 피처표·등급·운전상태를 코드화한다.
- 모델링 경쟁이 아니라 **데이터 진단**이다. 각 절은 "가설 → 실험 → 결과 → 모델 단계 반영" 순서로 적는다.
- 근거가 간접적인 판단은 **(추정)** 으로 표기한다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import data_quality as dq
import segments as sg
import signal_checks as sc
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
def save(name): plt.tight_layout(); plt.savefig(FIG / name, dpi=110); plt.close(); print("saved", name)

dfs = dq.load_all()
N, O = dfs["normal"], dfs["outlier"]
print(f"normal {N.shape}, outlier {O.shape}, normal seg {N.seg.nunique()}, outlier seg {O.seg.nunique()}")

normal (20000, 9), outlier (600, 9), normal seg 599, outlier seg 21


## 1. 세그먼트 등급화·운전 상태 (B-2 ①)

- **가설**: 이상 21세그먼트는 진동 세기로 3등급(확실 이상/경계/정상 유사)이 나뉘고, 정상 599세그먼트는 부하가 다른 여러 운전 상태로 나뉜다.
- **실험**: `segments.segment_features`로 세그먼트별 진동 AI0/AI1 RMS·피크·첨도·crest factor·DC 오프셋, 전류 RMS·피크·DC 오프셋을 만든다. 이상 세그먼트는 정상 세그먼트의 `vib_rms`(=√(AI0_rms²+AI1_rms²)) 분위수(중앙값 q50, 99%ile q99)로 등급을 매기고, 정상 세그먼트는 k-means(k=2~4)를 표준화된 피처에 적용해 실루엣 점수로 k를 고른다.

In [2]:
fN = sg.segment_features(N)
fO = sg.segment_features(O)
print("정상 세그먼트 피처표:", fN.shape, " 이상 세그먼트 피처표:", fO.shape)
display(fN.head(3))

정상 세그먼트 피처표: (599, 18)  이상 세그먼트 피처표: (21, 18)


,src,start,len,label,AI0_Vibration_rms,AI0_Vibration_peak,AI0_Vibration_kurt,AI0_Vibration_crest,AI0_Vibration_dc,AI1_Vibration_rms,AI1_Vibration_peak,AI1_Vibration_kurt,AI1_Vibration_crest,AI1_Vibration_dc,AI2_Current_rms,AI2_Current_peak,AI2_Current_dc,vib_rms
seg_id,,,,,,,,,,,,,,,,,,
0,normal,2022-07-12 00:00:00.019,41,0.0,0.069009,0.191580,0.169165,2.776144,-0.006957,0.163178,0.275627,-1.341630,1.689117,-0.009436,149.170481,219.93151,19.085886,0.177171
1,normal,2022-07-12 00:00:07.224,37,0.0,0.073404,0.136729,-0.938016,1.862683,0.007061,0.132412,0.244895,-1.310623,1.849498,0.006813,139.745235,217.53814,-5.676624,0.151397
2,normal,2022-07-12 00:00:15.267,24,0.0,0.040946,0.086432,-0.789943,2.110892,-0.010856,0.049729,0.093092,-0.948447,1.871972,-0.006255,80.082896,110.77358,-6.102717,0.064417


In [3]:
fO["grade"] = sg.grade_outlier_segments(fO, fN)
q50, q99 = fN["vib_rms"].quantile([.5, .99])
print(f"정상 vib_rms 분위수: q50={q50:.4f}, q99={q99:.4f}")
display(fO[["len", "AI0_Vibration_rms", "AI1_Vibration_rms", "vib_rms", "grade"]])
print(fO["grade"].value_counts())

정상 vib_rms 분위수: q50=0.1157, q99=0.2412


,len,AI0_Vibration_rms,AI1_Vibration_rms,vib_rms,grade
seg_id,,,,,
0,4,0.726850,0.465367,0.863062,확실 이상
1,50,0.391256,0.253066,0.465965,확실 이상
2,47,0.515215,0.290405,0.591424,확실 이상
3,31,0.045827,0.102027,0.111847,정상 유사
4,18,0.699241,0.297932,0.760067,확실 이상
5,3,0.964401,0.346671,1.024817,확실 이상
6,50,0.559658,0.254832,0.614944,확실 이상
7,40,0.337417,0.230235,0.408483,확실 이상
8,25,0.331365,0.260286,0.421369,확실 이상


grade
확실 이상    18
정상 유사     3
Name: count, dtype: int64


In [4]:
best_k, results, X = sg.choose_k_operating_states(fN)
for k, (labels, sil) in results.items():
    print(f"k={k}: silhouette={sil:.4f}, sizes={np.bincount(labels).tolist()}")
print(f"선택된 k = {best_k}")
fN["state"] = results[best_k][0]
summ = sg.operating_state_summary(fN)
display(summ)

k=2: silhouette=0.5991, sizes=[252, 347]
k=3: silhouette=0.4930, sizes=[279, 142, 178]
k=4: silhouette=0.4374, sizes=[148, 197, 137, 117]
선택된 k = 2


,AI0_Vibration_rms,AI1_Vibration_rms,AI2_Current_rms,AI2_Current_peak,n_segments,n_samples,share_samples
state,,,,,,,
0,0.084,0.164,156.344,229.196,252,8635,0.4318
1,0.055,0.060,86.446,120.705,347,11365,0.5682


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = {"확실 이상": "crimson", "경계": "orange", "정상 유사": "steelblue"}
order = fO.sort_values("start").index
ax.bar(range(len(order)), fO.loc[order, "vib_rms"], color=[colors[g] for g in fO.loc[order, "grade"]])
ax.axhline(q50, ls="--", c="steelblue", lw=1, label=f"정상 q50={q50:.3f}")
ax.axhline(q99, ls="--", c="crimson", lw=1, label=f"정상 q99={q99:.3f}")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=0)
ax.set_xlabel("이상 세그먼트 번호(시간순)"); ax.set_ylabel("vib_rms"); ax.set_title("이상 21세그먼트 등급 (정상 vib_rms 분위수 기준)")
ax.legend(fontsize=8)
save("02_segment_grades.png")

saved 02_segment_grades.png


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for s, marker in zip(sorted(fN["state"].unique()), ["o", "s", "^", "D"]):
    sub = fN[fN["state"] == s]
    axes[0].scatter(sub["AI2_Current_rms"], sub["vib_rms"], s=10, alpha=.5, label=f"state {s}", marker=marker)
axes[0].set_xlabel("AI2_Current_rms"); axes[0].set_ylabel("vib_rms"); axes[0].set_title("정상 운전 상태(k-means) — 전류 RMS vs 진동 RMS"); axes[0].legend(fontsize=8)
fN.sort_values("start")["state"].reset_index(drop=True).plot(ax=axes[1], lw=.6)
axes[1].set_title("세그먼트 순서(시간)에 따른 운전 상태"); axes[1].set_xlabel("세그먼트 순번"); axes[1].set_ylabel("state")
save("02_operating_states.png")

saved 02_operating_states.png


**1절 결과**: (노트북 실행 출력의 정확한 수치를 리포트에 인용한다.)

**모델 단계 반영**: `data_processed/segments.csv`(아래 셀에서 생성)를 GroupKFold의 그룹 키·soft label 원천으로 쓴다. `grade`는 이상 세그먼트의 soft label 후보(확실 이상=1.0, 경계=0.5, 정상 유사=0.0 등)로, `state`는 정상 세그먼트의 운전 상태 피처(또는 층화 기준)로 쓴다.

In [7]:
seg_table = sg.build_segments_table()
print("data_processed/segments.csv 저장:", seg_table.shape)
display(seg_table.head(2))
display(seg_table.tail(2))

data_processed/segments.csv 저장: (620, 22)


,seg_uid,seg_id,src,start,len,label,AI0_Vibration_rms,AI0_Vibration_peak,AI0_Vibration_kurt,AI0_Vibration_crest,AI0_Vibration_dc,AI1_Vibration_rms,AI1_Vibration_peak,AI1_Vibration_kurt,AI1_Vibration_crest,AI1_Vibration_dc,AI2_Current_rms,AI2_Current_peak,AI2_Current_dc,vib_rms,state,grade
0,normal_0,0,normal,2022-07-12 00:00:00.019,41,0.0,0.069009,0.191580,0.169165,2.776144,-0.006957,0.163178,0.275627,-1.341630,1.689117,-0.009436,149.170481,219.93151,19.085886,0.177171,0,<NA>
1,normal_1,1,normal,2022-07-12 00:00:07.224,37,0.0,0.073404,0.136729,-0.938016,1.862683,0.007061,0.132412,0.244895,-1.310623,1.849498,0.006813,139.745235,217.53814,-5.676624,0.151397,0,<NA>


,seg_uid,seg_id,src,start,len,label,AI0_Vibration_rms,AI0_Vibration_peak,AI0_Vibration_kurt,AI0_Vibration_crest,AI0_Vibration_dc,AI1_Vibration_rms,AI1_Vibration_peak,AI1_Vibration_kurt,AI1_Vibration_crest,AI1_Vibration_dc,AI2_Current_rms,AI2_Current_peak,AI2_Current_dc,vib_rms,state,grade
618,outlier_19,19,outlier,2022-07-17 10:53:42.668,10,1.0,0.032128,0.053349,-1.160567,1.660517,-0.018492,0.068152,0.151335,-0.619491,2.220554,0.023105,207.236039,301.59954,198.483489,0.075345,<NA>,정상 유사
619,outlier_20,20,outlier,2022-07-17 10:53:52.140,15,1.0,0.043272,0.069144,-0.718087,1.597899,-0.037587,0.021760,0.049116,-0.515344,2.257201,0.005571,183.383022,244.37907,180.085522,0.048435,<NA>,정상 유사


## 2. 신호 해상도·포화·DC 오프셋 누수 (B-2 ②)

- **가설**: (a) 채널값의 최소 간격으로 ADC 비트 수를 추정할 수 있다. (b) 전류 |값|이 특정 상한 근방에서 반복되면 클리핑(포화)이다. (c) 세그먼트 DC 오프셋(평균)만으로 normal/outlier가 갈리면 센서 재장착 등 누수 위험이다.
- **실험**: `signal_checks.resolution_table`(고유값 수·최소 간격·유일값 비율), `clipping_check`(|값| 최댓값 근방 반복 횟수), `dc_offset_auc`(세그먼트 DC 오프셋 컬럼 단독 AUC).

In [8]:
display(sc.resolution_table(dfs))

,src,channel,n_rows,n_unique,unique_ratio,min_step,range,bits_est
0,normal,AI0_Vibration,20000,19124,0.9562,1.000000e-06,0.6663,19.35
1,normal,AI1_Vibration,20000,19460,0.9730,1.000000e-06,0.7663,19.55
2,normal,AI2_Current,20000,19934,0.9967,1.100000e-04,544.8033,22.24
3,outlier,AI0_Vibration,600,594,0.9900,1.103800e-05,3.3059,18.19
4,outlier,AI1_Vibration,600,597,0.9950,1.157000e-05,1.2606,16.73
5,outlier,AI2_Current,600,340,0.5667,1.192090e+00,938.1772,9.62


In [9]:
display(sc.clipping_check(dfs, channel="AI2_Current"))
print("주의: outlier는 표본이 600행뿐이라 20,000행인 normal보다 최소 간격이 통계적으로 더 크게(비트 추정치가 낮게) 나올 수 있다 — 표본 크기 효과이지 실제 해상도 차이로 단정할 수 없다 (추정).")

,src,channel,max_abs,count_at_max,count_ge_99pct_of_max,share_ge_near_max
0,normal,AI2_Current,273.235,1,10,0.0005
1,outlier,AI2_Current,538.826,1,1,0.0017


주의: outlier는 표본이 600행뿐이라 20,000행인 normal보다 최소 간격이 통계적으로 더 크게(비트 추정치가 낮게) 나올 수 있다 — 표본 크기 효과이지 실제 해상도 차이로 단정할 수 없다 (추정).


In [10]:
dc_auc = sc.dc_offset_auc(fN, fO)
display(dc_auc)
print("DC 오프셋 AUC가 모두 0.5~0.53 수준이면 오프셋 단독으로는 라벨을 구분하지 못한다는 뜻 — 센서 재장착·설정 변경 누수의 직접 증거는 약하다는 의미.")

,column,auc,leakage_suspect
0,AI0_Vibration_dc,0.5181,False
1,AI1_Vibration_dc,0.5044,False
2,AI2_Current_dc,0.5290,False


DC 오프셋 AUC가 모두 0.5~0.53 수준이면 오프셋 단독으로는 라벨을 구분하지 못한다는 뜻 — 센서 재장착·설정 변경 누수의 직접 증거는 약하다는 의미.


**2절 결과**: (수치는 위 표 출력에서 인용)

**모델 단계 반영**: DC 오프셋 AUC가 낮으면(<0.6) 세그먼트 평균 제거를 **필수 전처리로 강제하지 않는다** — 다만 다른 날짜에 재수집 시 센서 드리프트가 생길 수 있으므로 강건성 차원의 옵션(제거 vs 유지 두 버전 비교)으로 남긴다. 전류 채널은 클리핑 근접 비율이 낮아(포화가 흔치 않아) RMS/피크를 그대로 사용해도 된다.

## 3. 에일리어싱 검증 (B-2 ③)

- **가설**: 전류는 60 Hz AC가 10 Hz 샘플링으로 접힌(에일리어싱) 단일 저주파 성분만 가지고, 진동 채널에는 전류와 같은 주기의 전원 노이즈가 섞이지 않는다.
- **실험**: (a) 세그먼트별 zero-crossing 주기의 평균·분산. (b) 60±α Hz 대역을 훑어 관측된 에일리어싱 주파수에 가장 잘 맞는 실제 주파수 후보 계산. (c) 대표 세그먼트 FFT로 피크가 하나뿐인지 확인. (d) 진동 채널 자기상관·zero-crossing과 전류 채널의 상관.

In [11]:
zc_cur = sc.zero_crossing_by_segment(N, "AI2_Current", min_len=10).replace([np.inf, -np.inf], np.nan).dropna()
print(f"normal 전류 zero-crossing 주기(샘플): mean={zc_cur.mean():.3f}, sd={zc_cur.std():.3f}, n_seg={len(zc_cur)}")
print(f"  → 초 단위 mean={zc_cur.mean()/10:.3f}s, 겉보기 주파수={10/zc_cur.mean():.4f} Hz")
zc_cur_o = sc.zero_crossing_by_segment(O, "AI2_Current", min_len=10).replace([np.inf, -np.inf], np.nan).dropna()
print(f"outlier 전류 zero-crossing 주기(샘플): mean={zc_cur_o.mean():.3f}, sd={zc_cur_o.std():.3f}, n_seg(유효)={len(zc_cur_o)} / 전체 21")
n_inf = sc.zero_crossing_by_segment(O, "AI2_Current", min_len=10).replace([np.inf,-np.inf], np.nan).isna().sum()
print(f"outlier: 부호 변화가 아예 없어(zero-crossing=0) 주기를 못 구한 세그먼트 {n_inf}개 — 이상 구간에서는 전류의 규칙적 AC 성분이 흐트러짐을 시사 (추정)")

normal 전류 zero-crossing 주기(샘플): mean=17.582, sd=2.723, n_seg=530
  → 초 단위 mean=1.758s, 겉보기 주파수=0.5687 Hz
outlier 전류 zero-crossing 주기(샘플): mean=10.783, sd=7.444, n_seg(유효)=12 / 전체 21
outlier: 부호 변화가 아예 없어(zero-crossing=0) 주기를 못 구한 세그먼트 5개 — 이상 구간에서는 전류의 규칙적 AC 성분이 흐트러짐을 시사 (추정)


In [12]:
segN = dq.segment_table(N)
long_segs = segN[segN.n == 50].index.tolist()
rep_seg = long_segs[0]
g = N[N.seg == rep_seg]
freqs, amp = sc.fft_spectrum(g["AI2_Current"].to_numpy(), fs=10.0)
peak_idx = np.argsort(amp[1:])[::-1][:5] + 1
peak_freq = freqs[peak_idx[0]]
print("대표 세그먼트(정상, 50샘플) FFT 상위 5개 성분:")
for i in peak_idx:
    print(f"  freq={freqs[i]:.4f} Hz  amp={amp[i]:.2f}")
print(f"\n최댓값 성분 freq={peak_freq:.4f} Hz, 2위 대비 진폭비={amp[peak_idx[0]]/amp[peak_idx[1]]:.1f}배 → 사실상 단일 성분")
cands = sc.alias_candidates(peak_freq, fs=10.0, true_freq_range=(55, 65), step=0.1)
print("\n60Hz 부근 실제 주파수 후보 (관측 에일리어싱 주파수에 가장 근접한 순):")
display(cands.head(6))

대표 세그먼트(정상, 50샘플) FFT 상위 5개 성분:


  freq=0.6000 Hz  amp=57.52
  freq=1.8000 Hz  amp=3.32
  freq=3.0000 Hz  amp=1.67
  freq=4.2000 Hz  amp=1.41
  freq=0.4000 Hz  amp=0.18

최댓값 성분 freq=0.6000 Hz, 2위 대비 진폭비=17.3배 → 사실상 단일 성분

60Hz 부근 실제 주파수 후보 (관측 에일리어싱 주파수에 가장 근접한 순):


,f_true_hz,n,alias_freq_hz,diff_from_observed
0,60.6,6,0.6,0.0
1,59.4,6,0.6,0.0
2,59.3,6,0.7,0.1
3,59.5,6,0.5,0.1
4,60.7,6,0.7,0.1
5,60.5,6,0.5,0.1


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(g["ts"], g["AI2_Current"], marker=".", lw=.8)
axes[0].set_title(f"대표 정상 세그먼트(seg {rep_seg}, {len(g)}샘플) 파형")
axes[1].stem(freqs, amp, basefmt=" ")
axes[1].set_xlim(0, 5); axes[1].set_xlabel("Hz"); axes[1].set_ylabel("amplitude")
axes[1].set_title(f"FFT — 피크는 {peak_freq:.2f} Hz 근방 하나뿐 (에일리어싱 주기)")
save("02_fft_alias.png")

saved

 02_fft_alias.png


In [14]:
ac_cur = sc.autocorr(g["AI2_Current"].to_numpy(), max_lag=20)
for ch in ["AI0_Vibration", "AI1_Vibration"]:
    ac_v = sc.autocorr(g[ch].to_numpy(), max_lag=20)
    zc_v = dq.zero_crossing_period(g[ch].to_numpy())
    print(f"{ch}: zero-crossing 주기(대표세그먼트)={zc_v:.2f}샘플, lag1 자기상관={ac_v[1]:.3f}")
print(f"AI2_Current: zero-crossing 주기(대표세그먼트)={dq.zero_crossing_period(g['AI2_Current'].to_numpy()):.2f}샘플, lag1 자기상관={ac_cur[1]:.3f}")

rows = []
for seg_id, gg in N.groupby("seg"):
    if len(gg) < 20:
        continue
    rows.append((dq.zero_crossing_period(gg["AI2_Current"].to_numpy()),
                 dq.zero_crossing_period(gg["AI0_Vibration"].to_numpy()),
                 dq.zero_crossing_period(gg["AI1_Vibration"].to_numpy())))
per_seg = pd.DataFrame(rows, columns=["pc", "p0", "p1"]).replace([np.inf, -np.inf], np.nan).dropna()
print(f"\n세그먼트별(n={len(per_seg)}) 전류-진동 zero-crossing 주기 상관: AI0 r={per_seg['pc'].corr(per_seg['p0']):.4f}, AI1 r={per_seg['pc'].corr(per_seg['p1']):.4f}")
print("상관이 0에 가까우면 진동 채널의 주기 구조가 전류(전원 유래 성분)와 무관 → 전원 노이즈 혼입 근거 약함")

AI0_Vibration: zero-crossing 주기(대표세그먼트)=5.26샘플, lag1 자기상관=0.338
AI1_Vibration: zero-crossing 주기(대표세그먼트)=4.00샘플, lag1 자기상관=-0.088
AI2_Current: zero-crossing 주기(대표세그먼트)=16.67샘플, lag1 자기상관=0.911

세그먼트별(n=452) 전류-진동 zero-crossing 주기 상관: AI0 r=-0.0136, AI1 r=0.0036
상관이 0에 가까우면 진동 채널의 주기 구조가 전류(전원 유래 성분)와 무관 → 전원 노이즈 혼입 근거 약함


**3절 결과**: (수치는 위 출력에서 인용)

**모델 단계 반영**: FFT·주파수 도메인 피처는 **사용하지 않는다**(에일리어싱으로 무의미). 시간영역 진폭 피처(RMS·피크·첨도·crest factor)만 사용. 진동 채널에 전원 노이즈 혼입 근거가 약하므로 별도 노치 필터는 불필요.

## 4. 이상 구간의 시간 진행 (B-2 ④)

- **가설**: 이상 21세그먼트를 시간순으로 보면 점진적으로 악화되는 추세가 있을 수 있다(조기탐지 스토리). 또한 600행이 정확히 12×50(=12개 완전한 burst)인지, 첫·끝 세그먼트가 잘렸는지 확인한다.
- **실험**: `segments.outlier_trend`로 시간순 `vib_rms`의 선형 기울기·Spearman 상관을 구하고, 세그먼트 길이 분포로 절단 여부를 본다.

In [15]:
trend = sg.outlier_trend(fO, col="vib_rms")
print(trend)
print(f"\n세그먼트 길이 합계: {fO['len'].sum()} (12×50={12*50}과 비교) — 21개 세그먼트, 길이 최소 {fO['len'].min()}~최대 {fO['len'].max()}")
print(f"첫 세그먼트 길이 {fO.sort_values('start')['len'].iloc[0]}, 마지막 세그먼트 길이 {fO.sort_values('start')['len'].iloc[-1]} — 둘 다 50 미만이면 절단 가능성")
display(fO.sort_values("start")[["len", "vib_rms", "grade"]])

{'slope': -0.01842623966811663, 'intercept': 0.6940607405591279, 'spearman_rho': -0.42987012987012985, 'spearman_p': 0.05178188171189559}

세그먼트 길이 합계: 600 (12×50=600과 비교) — 21개 세그먼트, 길이 최소 3~최대 50
첫 세그먼트 길이 4, 마지막 세그먼트 길이 15 — 둘 다 50 미만이면 절단 가능성


,len,vib_rms,grade
seg_id,,,
0,4,0.863062,확실 이상
1,50,0.465965,확실 이상
2,47,0.591424,확실 이상
3,31,0.111847,정상 유사
4,18,0.760067,확실 이상
5,3,1.024817,확실 이상
6,50,0.614944,확실 이상
7,40,0.408483,확실 이상
8,25,0.421369,확실 이상


In [16]:
fig, ax = plt.subplots(figsize=(9, 4))
fo_sorted = fO.sort_values("start").reset_index()
t = np.arange(len(fo_sorted))
ax.plot(t, fo_sorted["vib_rms"], marker="o")
z = np.polyfit(t, fo_sorted["vib_rms"], 1)
slope_txt = f"선형 추세 (기울기={trend['slope']:.4f}/세그먼트)"
ax.plot(t, np.polyval(z, t), "--", c="crimson", label=slope_txt)
for i, row in fo_sorted.iterrows():
    if row["grade"] != "확실 이상":
        ax.annotate(row["grade"], (i, row["vib_rms"]), fontsize=8, color="steelblue")
title_txt = f"이상 21세그먼트 시간 진행 (Spearman rho={trend['spearman_rho']:.3f}, p={trend['spearman_p']:.3f})"
ax.set_xlabel("이상 세그먼트 순번(시간순)"); ax.set_ylabel("vib_rms"); ax.set_title(title_txt)
ax.legend(fontsize=8)
save("02_outlier_trend.png")

saved 02_outlier_trend.png


**4절 결과**: (수치는 위 출력에서 인용) 600행이 12×50이 아니라 길이가 3~50까지 흩어진 21개 burst의 합임을 확인했다(아래 표 참고). 첫·끝 세그먼트 길이가 50 미만이면 관측 구간이 이벤트 전체가 아니라 잘린 일부일 가능성이 있다(추정).

**모델 단계 반영**: 추세 기울기·Spearman p값이 유의하지 않으면(예: p>0.1) "점진 악화형 조기탐지"보다는 "발생 즉시 탐지" 문제로 재정의하고, 현장 활용안에서도 예지(prognosis)보다 즉시 경보 설계를 우선한다.

## 5. burst 수집 규칙 (B-2 ⑤)

- **가설**: burst는 주기적 폴링(일정 간격)으로 수집되며, 이 간격이 탐지 지연의 물리적 하한을 결정한다.
- **실험**: 세그먼트 시작 시각 간격 분포, 총 가동시간 대비 duty cycle.

In [17]:
starts_n = N.groupby("seg")["ts"].min().sort_values()
iv_n = starts_n.diff().dt.total_seconds().dropna()
print("normal burst 시작 간격(초):"); display(iv_n.describe())
print(f"중앙값={iv_n.median():.3f}s, IQR=[{iv_n.quantile(.25):.3f}, {iv_n.quantile(.75):.3f}]")

starts_o = O.groupby("seg")["ts"].min().sort_values()
iv_o = starts_o.diff().dt.total_seconds().dropna()
print("\noutlier burst 시작 간격(초):"); display(iv_o.describe())

dur_sum = dq.segment_table(N)["dur_s"].sum()
elapsed = (N["ts"].max() - N["ts"].min()).total_seconds()
print(f"\nnormal 총 가동(burst 내부) 시간 {dur_sum:.1f}s / 전체 경과 {elapsed:.1f}s → duty cycle {dur_sum/elapsed:.4f}")
print(f"burst 간격 중앙값 {iv_n.median():.2f}s ≈ 다음 burst가 와야 값을 볼 수 있는 시간 → 규칙 기반 탐지 지연의 물리적 하한")

normal burst 시작 간격(초):


count    598.000000
mean       7.711554
std        1.698796
min        1.545000
25%        7.299000
50%        7.958000
75%        8.057750
max       17.643000
Name: ts, dtype: float64

중앙값=7.958s, IQR=[7.299, 8.058]

outlier burst 시작 간격(초):


count    20.000000
mean      8.209850
std       0.732269
min       6.680000
25%       8.071750
50%       8.129500
75%       8.486000
max       9.855000
Name: ts, dtype: float64


normal 총 가동(burst 내부) 시간 1940.0s / 전체 경과 4615.8s → duty cycle 0.4203


burst 간격 중앙값 7.96s ≈ 다음 burst가 와야 값을 볼 수 있는 시간 → 규칙 기반 탐지 지연의 물리적 하한


**5절 결과**: (수치는 위 출력에서 인용)

**모델 단계 반영**: 현장 활용안의 "탐지 지연" 수치를 보고할 때 burst 간격 중앙값을 **구조적 하한**으로 같이 명시한다(신호 처리로는 더 줄일 수 없음). duty cycle이 낮다는 것은 실제 신호 취득이 간헐적이라는 뜻이므로, 실시간 연속 모니터링이 아니라 "주기적 스냅샷 점검" 프레임으로 현장 활용안을 서술한다.

## 6. 채널 간 관계 (B-2 ⑥)

- **가설**: 정상 구간에서는 부하(전류)와 진동이 비례하므로, 부하로 보정한 진동 지표(진동 RMS/전류 RMS)가 원시 진동 RMS보다 정상/이상 분리에 유리하거나, 적어도 부하 변화로 인한 오경보를 줄일 수 있다.
- **실험**: 세그먼트별 AI0-AI1 원시신호 상관, 전류 RMS-진동 RMS 산점(정상/이상), `vib_rms` 단독 AUC vs 부하보정지표(`vib_rms/AI2_Current_rms`) 단독 AUC 비교.

In [18]:
corr01 = sg.within_segment_corr(N, "AI0_Vibration", "AI1_Vibration", min_len=5)
print(f"정상 세그먼트별 AI0-AI1 상관(n={len(corr01)}): mean={corr01.mean():.4f}, median={corr01.median():.4f}")
print(f"normal 전류RMS-진동RMS(vib_rms) 상관: {fN['AI2_Current_rms'].corr(fN['vib_rms']):.4f}")

fN["load_corr"] = fN["vib_rms"] / fN["AI2_Current_rms"]
fO["load_corr"] = fO["vib_rms"] / fO["AI2_Current_rms"]
auc_vib = sg.separability_auc(fN, fO, "vib_rms")
auc_loadcorr = sg.separability_auc(fN, fO, "load_corr")
print(f"\nvib_rms 단독 AUC = {auc_vib:.4f}")
print(f"load_corr(vib_rms/current_rms) 단독 AUC = {auc_loadcorr:.4f}")
print("부하보정 지표가 원시 vib_rms보다 AUC가 낮으면, 이 데이터에서는 부하 보정이 분리력을 개선하지 못한다는 뜻 (추정: 이상 구간의 진동 증가폭이 워낙 커서 원시값만으로도 이미 포화 수준으로 분리됨)")

정상 세그먼트별 AI0-AI1 상관(n=570): mean=0.2334, median=0.2803
normal 전류RMS-진동RMS(vib_rms) 상관: 0.8913

vib_rms 단독 AUC = 0.8946
load_corr(vib_rms/current_rms) 단독 AUC = 0.8629
부하보정 지표가 원시 vib_rms보다 AUC가 낮으면, 이 데이터에서는 부하 보정이 분리력을 개선하지 못한다는 뜻 (추정: 이상 구간의 진동 증가폭이 워낙 커서 원시값만으로도 이미 포화 수준으로 분리됨)


In [19]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(fN["AI2_Current_rms"], fN["vib_rms"], s=10, alpha=.4, label="normal", c="steelblue")
ax.scatter(fO["AI2_Current_rms"], fO["vib_rms"], s=25, alpha=.8, label="outlier", c="crimson", marker="x")
ax.set_xlabel("AI2_Current_rms"); ax.set_ylabel("vib_rms"); ax.set_title("세그먼트별 전류 RMS - 진동 RMS 관계")
ax.legend()
save("02_channel_relation.png")

saved 02_channel_relation.png


**6절 결과**: (수치는 위 출력에서 인용)

**모델 단계 반영**: `load_corr`가 `vib_rms`보다 낫지 않다면 주 피처는 원시 `vib_rms`(및 AI0/AI1 개별 RMS)로 두고, `load_corr`는 "부하 변화에 따른 오경보 억제"가 필요한 운전 상태(예: 저부하 state)에 한정된 **보조 규칙**으로만 쓴다. AI0-AI1 상관이 중간 정도(0.2~0.4대)이면 두 채널을 평균하지 않고 **개별 피처로 유지**한다(서로 다른 정보가 있다는 뜻).

## 7. 윈도우 길이·정상량 민감도 (B-2 ⑦)

- **가설**: 윈도우가 길수록(1→5초) 진동 RMS의 정상/이상 분리력(AUC)이 좋아지지만, 전류 RMS는 에일리어싱 때문에 윈도우를 늘려도 크게 개선되지 않는다. 또한 임계값을 정상 데이터 앞부분 일부(20~80%)만으로 잡아도 오경보율이 크게 늘지 않으면, 적은 정상 데이터로도 규칙을 세울 수 있다.
- **실험**: `segments.window_auc_table`(세그먼트 경계를 넘지 않는 `dq.rolling_rms` 재사용, 1/2/3/5초), `segments.leakage_free_fpr_table`(시간순 앞 20/40/60/80% 세그먼트로 99분위 임계값을 잡고 나머지에서 초과 비율).

In [20]:
wt = sg.window_auc_table(N, O, windows_sec=(1, 2, 3, 5))
piv = wt.pivot(index="window_sec", columns="channel", values="auc")
display(piv)

channel,AI0_Vibration,AI1_Vibration,AI2_Current
window_sec,,,
1,0.8067,0.7913,0.5194
2,0.8771,0.8509,0.5023
3,0.9182,0.8876,0.5289
5,1.0000,0.9782,0.6347


In [21]:
fig, ax = plt.subplots(figsize=(7, 4))
for ch in dq.SENSORS:
    sub = wt[wt.channel == ch]
    ax.plot(sub["window_sec"], sub["auc"], marker="o", label=ch)
ax.axhline(0.5, ls=":", c="gray")
ax.set_xlabel("윈도우 길이(초)"); ax.set_ylabel("AUC (normal vs outlier)"); ax.set_title("윈도우 길이별 이동 RMS AUC")
ax.legend()
save("02_window_auc.png")

saved 02_window_auc.png


In [22]:
fpr_tab = sg.leakage_free_fpr_table(N, win=10, channel="AI0_Vibration", fractions=(0.2, 0.4, 0.6, 0.8))
display(fpr_tab)
print("해석: train_frac이 작아도(정상 20%만 써도) fpr이 크게 뛰지 않으면, 초기 짧은 정상 구간만으로도 임계값을 잠정 확정할 수 있다는 뜻.")

,train_frac,n_train_seg,n_test_seg,threshold_q99,n_test_samples,fpr
0,0.2,120,479,0.1251,11521,0.0147
1,0.4,240,359,0.1270,8745,0.0134
2,0.6,359,240,0.1277,5964,0.0132
3,0.8,479,120,0.1320,3043,0.0023


해석: train_frac이 작아도(정상 20%만 써도) fpr이 크게 뛰지 않으면, 초기 짧은 정상 구간만으로도 임계값을 잠정 확정할 수 있다는 뜻.


**7절 결과**: (수치는 위 표에서 인용)

**모델 단계 반영**: 윈도우 길이는 **1~3초**(AUC가 이미 충분히 높고 탐지 지연도 짧음) 사이에서 현장 활용안의 지연 허용치에 맞춰 고른다. 전류 채널은 윈도우를 늘려도 개선이 작으므로 진동 채널을 주 피처로, 전류는 보조(부하 지표)로만 쓴다. 정상 20%만으로 잡은 임계값의 오경보율이 표의 값 수준이면, 콜드스타트(수집 초기)에도 규칙을 쓸 수 있다는 근거로 리포트에 남긴다.

## 8. 규칙 기반 탐지 지연 (B-2 ⑧)

- **가설**: 정상 전체의 99분위 임계값을 넘는 첫 샘플까지 걸리는 시간(지연)은 대부분 1초 안팎이고, 조용한 세그먼트(3·19·20)는 임계값을 넘지 못해 규칙 기반으로는 미탐지된다.
- **실험**: `segments.detection_delay_table`로 1초 윈도우(AI0_Vibration) 기준 세그먼트별 최초 초과 위치를 구한다. 세그먼트 길이가 윈도우보다 짧아 판정 자체가 불가능한 경우와, 조용해서 미탐지인 경우를 구분한다.

In [23]:
thr_full = dq.rolling_rms(N, win=10)["AI0_Vibration"].dropna().quantile(0.99)
print(f"정상 전체(1초 윈도우, AI0_Vibration) q99 임계값 = {thr_full:.4f}")
delay_tab = sg.detection_delay_table(O, threshold=thr_full, win=10, channel="AI0_Vibration")
display(delay_tab)
print(delay_tab["reason"].value_counts())
det = delay_tab[delay_tab.detected]
print(f"\n탐지된 세그먼트 {len(det)}/{len(delay_tab)}, 지연(샘플) 중앙값={det['delay_samples'].median():.1f} (={det['delay_samples'].median()/10:.2f}s), 평균={det['delay_samples'].mean():.2f}")

정상 전체(1초 윈도우, AI0_Vibration) q99 임계값 = 0.1296


,seg,n,n_valid_windows,detected,delay_samples,delay_sec,reason
0,0,4,0,False,NaN,NaN,세그먼트 길이<윈도우(판정 불가)
1,1,50,41,True,9.0,0.9,탐지
2,2,47,38,True,13.0,1.3,탐지
3,3,31,22,False,NaN,NaN,조용한 세그먼트(라벨 노이즈 의심)
4,4,18,9,True,9.0,0.9,탐지
5,5,3,0,False,NaN,NaN,세그먼트 길이<윈도우(판정 불가)
6,6,50,41,True,9.0,0.9,탐지
7,7,40,31,True,9.0,0.9,탐지
8,8,25,16,True,17.0,1.7,탐지
9,9,8,0,False,NaN,NaN,세그먼트 길이<윈도우(판정 불가)


reason
탐지                     14
세그먼트 길이<윈도우(판정 불가)      4
조용한 세그먼트(라벨 노이즈 의심)     3
Name: count, dtype: int64

탐지된 세그먼트 14/21, 지연(샘플) 중앙값=9.0 (=0.90s), 평균=11.64


In [24]:
fig, ax = plt.subplots(figsize=(8, 4))
reason_colors = {"탐지": "steelblue", "조용한 세그먼트(라벨 노이즈 의심)": "orange", "세그먼트 길이<윈도우(판정 불가)": "gray", "미탐지": "crimson"}
for reason, sub in delay_tab.groupby("reason"):
    y = sub["delay_samples"].fillna(-2)
    ax.scatter(sub["seg"], y, label=reason, c=reason_colors.get(reason, "black"), s=40)
ax.set_xlabel("이상 세그먼트 번호"); ax.set_ylabel("탐지 지연(샘플, 1초윈도우=10샘플/초)")
ax.set_title("세그먼트별 규칙 기반 탐지 지연")
ax.legend(fontsize=8, loc="upper left")
save("02_detection_delay.png")

saved

 02_detection_delay.png


**8절 결과**: (수치는 위 출력에서 인용) 미탐지 세그먼트 중 3·19·20은 진동이 실제로 조용해 라벨 노이즈로 의심되는 것과, 나머지는 세그먼트 길이가 1초 윈도우(10샘플)보다 짧아 애초에 판정 불가능한 경우로 나뉜다 — 이 둘은 원인이 다르므로 오류분석에서 같이 묶으면 안 된다.

**모델 단계 반영**: 지표 정의를 "세그먼트당 탐지 지연(중앙값, 샘플/초)"과 "판정 불가 세그먼트 비율(길이<윈도우)"로 분리해서 보고한다. 윈도우보다 짧은 세그먼트가 있다는 사실 자체가 윈도우 길이 선택(7절)에 대한 하한 제약이 된다.

## 9. 모델 단계 반영 사항 요약

| 항목 | 규칙 |
|---|---|
| split 키 | 세그먼트(`seg_uid`) 단위 GroupKFold. 정상은 추가로 시간순 블록(앞/뒤) 분할 병행 |
| soft label | 이상 세그먼트 `grade` — 확실 이상=1.0, 경계=0.5, 정상 유사=0.0(라벨 노이즈로 별도 보고) |
| 전처리(오프셋) | DC 오프셋 단독 AUC가 낮으면(2절) 세그먼트 평균 제거는 선택 사항(강건성 비교용) |
| 윈도우 길이 | 1~3초 (7절 AUC·8절 탐지 지연 근거) |
| 주 피처 | AI0/AI1 진동 RMS·피크·첨도·crest factor (시간영역만, FFT 금지) |
| 보조 피처 | 전류 RMS·피크, `load_corr`(부하보정 진동지표)는 저부하 상태 한정 보조 규칙 |
| 지표 정의 | 세그먼트 단위 탐지율·오경보율, 탐지 지연(중앙값, 샘플/초), 판정 불가 세그먼트 비율(길이<윈도우) 별도 보고 |

리포트 `reports/02_diagnosis_deep.md`에 위 표와 세부 수치를 정리한다.